# Clasificación de Géneros de Escarabajos con ResNet50

Este notebook implementa un modelo de clasificación de géneros de escarabajos usando una red neuronal convolucional preentrenada (ResNet50) y PyTorch.

## Estructura del Proyecto
- Dataset: 32 géneros de escarabajos
- Imágenes: `cropped_beetles/images/genero_n/`
- Etiquetas: `cropped_beetles/labels/genero_n/`

In [1]:
# Configuración del entorno para evitar problemas de compatibilidad
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# Verificar versiones instaladas
try:
    import numpy as np
    import torch
    import scipy
    import sklearn
    
    print(f"NumPy version: {np.__version__}")
    print(f"PyTorch version: {torch.__version__}")
    print(f"SciPy version: {scipy.__version__}")
    print(f"Scikit-learn version: {sklearn.__version__}")
    
    if hasattr(torch, 'cuda') and torch.cuda.is_available():
        print(f"CUDA available: {torch.cuda.is_available()}")
        print(f"CUDA version: {torch.version.cuda}")
        
    # Verificar compatibilidad de versiones
    import pkg_resources
    from packaging import version
    
    np_version = version.parse(np.__version__)
    min_np_version = version.parse("1.25.2")
    max_np_version = version.parse("2.6.0")
    
    if np_version < min_np_version or np_version >= max_np_version:
        print(f"\n⚠️ ADVERTENCIA: La versión actual de NumPy ({np.__version__}) no es compatible con SciPy.")
        print(f"   Se recomienda una versión entre 1.25.2 y 2.6.0\n")
        print("Comandos para actualizar (ejecutar en terminal):")
        print("conda activate beetles")
        print("conda install numpy=1.25.2 -y")
except ImportError as e:
    print(f"Error al importar: {e}")
    print("\nEs posible que necesites ejecutar estos comandos en una terminal:")
    print("conda activate beetles")
    print("conda install numpy=1.25.2 scipy scikit-learn -y")

NumPy version: 1.25.2
PyTorch version: 2.5.1
SciPy version: 1.16.0
Scikit-learn version: 1.7.2
CUDA available: True
CUDA version: 12.1


C:\Users\Estudiantes\AppData\Local\Temp\ipykernel_12976\1655141021.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score


In [ ]:

# Configurar la semilla para reproducibilidad
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Verificar si GPU está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilizando dispositivo: {device}")

In [ ]:
# Definir la clase para nuestro conjunto de datos de escarabajos
class BeetleDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# Función para cargar las imágenes y etiquetas
def load_dataset(data_dir='cropped_beetles/images'):
    image_paths = []
    labels = []
    class_names = []
    
    # Obtener las carpetas de géneros (clases)
    for class_idx, genus_folder in enumerate(sorted(os.listdir(data_dir))):
        genus_path = os.path.join(data_dir, genus_folder)
        
        if os.path.isdir(genus_path):
            class_names.append(genus_folder)
            
            # Obtener todas las imágenes de cada género
            for img_file in os.listdir(genus_path):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    img_path = os.path.join(genus_path, img_file)
                    image_paths.append(img_path)
                    labels.append(class_idx)
    
    print(f"Total de clases: {len(class_names)}")
    print(f"Total de imágenes: {len(image_paths)}")
    
    return image_paths, labels, class_names

# Cargar el dataset
image_paths, labels, class_names = load_dataset()

# Mostrar la distribución de clases
unique_labels, counts = np.unique(labels, return_counts=True)
plt.figure(figsize=(15, 5))
plt.bar([class_names[i] for i in unique_labels], counts)
plt.xticks(rotation=90)
plt.xlabel('Géneros')
plt.ylabel('Número de imágenes')
plt.title('Distribución de imágenes por género')
plt.tight_layout()
plt.show()

# Definir transformaciones para preprocesamiento de imágenes
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Dividir los datos en conjuntos de entrenamiento y validación
# Usaremos una división estratificada para mantener la proporción de clases
train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths, labels, test_size=0.2, random_state=seed, stratify=labels
)

print(f"Imágenes de entrenamiento: {len(train_paths)}")
print(f"Imágenes de validación: {len(val_paths)}")

# Calcular pesos para manejar el desbalance de clases
class_counts = np.bincount(train_labels)
class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)
sample_weights = class_weights[train_labels]

# Crear un WeightedRandomSampler para el conjunto de entrenamiento
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_labels),
    replacement=True
)

# Crear conjuntos de datos
train_dataset = BeetleDataset(train_paths, train_labels, transform=transform_train)
val_dataset = BeetleDataset(val_paths, val_labels, transform=transform_val)

# Crear data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    sampler=sampler,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

# Visualizar algunas imágenes de entrenamiento para verificar
def show_batch(data_loader):
    for images, labels in data_loader:
        fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(12, 6))
        axes = axes.flatten()
        
        for i, (img, lbl) in enumerate(zip(images[:8], labels[:8])):
            img = img.permute(1, 2, 0).numpy()
            img = (img * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
            img = np.clip(img, 0, 1)
            
            axes[i].imshow(img)
            axes[i].set_title(f'Clase: {class_names[lbl]}')
            axes[i].axis('off')
            
        plt.tight_layout()
        plt.show()
        break

show_batch(train_loader)

In [ ]:
# Crear el modelo ResNet50 preentrenado y modificar la capa final
def create_model(num_classes):
    # Cargar el modelo preentrenado
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    
    # Congelar todos los parámetros
    for param in model.parameters():
        param.requires_grad = False
    
    # Obtener el número de características del último layer
    num_ftrs = model.fc.in_features
    
    # Reemplazar el último layer completamente conectado
    model.fc = nn.Sequential(
        nn.Linear(num_ftrs, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes)
    )
    
    # Descongelar las últimas capas para fine-tuning
    for child in list(model.children())[-3:]:
        for param in child.parameters():
            param.requires_grad = True
            
    return model

# Crear el modelo
num_classes = len(class_names)
model = create_model(num_classes)
model = model.to(device)

# Mostrar la arquitectura del modelo
print(model)

In [ ]:
# Definir la función de pérdida y el optimizador
# Usamos CrossEntropyLoss con pesos de clase para manejar el desbalance
class_weights_tensor = class_weights.to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

# Usar un planificador de tasa de aprendizaje para reducir la tasa cuando la validación se estanca
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.1, verbose=True)

# Función para entrenar el modelo
def train_model(model, criterion, optimizer, scheduler, train_loader, val_loader, num_epochs=15):
    # Listas para almacenar métricas de entrenamiento
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    
    # Mejor precisión de validación para guardar el mejor modelo
    best_val_acc = 0.0
    
    for epoch in range(num_epochs):
        # Fase de entrenamiento
        model.train()
        running_loss = 0.0
        running_corrects = 0
        total = 0
        
        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            # Poner a cero los gradientes
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
            
            # Backward pass y optimización
            loss.backward()
            optimizer.step()
            
            # Estadísticas
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            total += labels.size(0)
            
        epoch_loss = running_loss / total
        epoch_acc = running_corrects.double() / total
        train_losses.append(epoch_loss)
        train_accs.append(epoch_acc.item())
        
        # Fase de validación
        model.eval()
        running_loss = 0.0
        running_corrects = 0
        total = 0
        
        # No calcular gradientes en la validación
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                # Forward pass
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)
                
                # Estadísticas
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                total += labels.size(0)
                
        val_loss = running_loss / total
        val_acc = running_corrects.double() / total
        val_losses.append(val_loss)
        val_accs.append(val_acc.item())
        
        # Paso del planificador
        scheduler.step(val_loss)
        
        # Imprimir estadísticas por época
        print(f'Época {epoch+1}/{num_epochs}: '
              f'Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.4f}, '
              f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')
        
        # Guardar el mejor modelo
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_beetle_model.pth')
            print(f'Nuevo mejor modelo guardado con precisión: {val_acc:.4f}')
    
    # Cargar el mejor modelo
    model.load_state_dict(torch.load('best_beetle_model.pth'))
    
    return model, train_losses, val_losses, train_accs, val_accs

# Entrenar el modelo
model, train_losses, val_losses, train_accs, val_accs = train_model(
    model, criterion, optimizer, scheduler, train_loader, val_loader
)

In [ ]:
# Visualizar la pérdida y precisión durante el entrenamiento
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Entrenamiento')
plt.plot(val_losses, label='Validación')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.title('Pérdida durante el entrenamiento')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Entrenamiento')
plt.plot(val_accs, label='Validación')
plt.xlabel('Época')
plt.ylabel('Precisión')
plt.title('Precisión durante el entrenamiento')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Evaluar el modelo en el conjunto de validación
def evaluate_model(model, data_loader):
    model.eval()
    predictions = []
    true_labels = []
    
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    
    return true_labels, predictions

# Obtener predicciones
true_labels, predictions = evaluate_model(model, val_loader)

# Calcular métricas
accuracy = accuracy_score(true_labels, predictions)
precision = precision_score(true_labels, predictions, average='weighted')
recall = recall_score(true_labels, predictions, average='weighted')
f1 = f1_score(true_labels, predictions, average='weighted')

print(f"Precisión: {accuracy:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print("\nReporte de clasificación:")
print(classification_report(true_labels, predictions, target_names=class_names))

In [ ]:
# Visualizar la matriz de confusión
plt.figure(figsize=(15, 15))
cm = confusion_matrix(true_labels, predictions)
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Matriz de Confusión')
plt.colorbar()

# Etiquetas para los ejes
tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=90)
plt.yticks(tick_marks, class_names)

# Añadir valores numéricos a la matriz
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j],
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

plt.tight_layout()
plt.ylabel('Etiqueta Real')
plt.xlabel('Predicción')
plt.show()

In [ ]:
# Visualizar algunas predicciones en imágenes de validación
def visualize_predictions(model, data_loader, num_images=10):
    model.eval()
    images_so_far = 0
    fig = plt.figure(figsize=(15, 10))
    
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            for j in range(inputs.size()[0]):
                images_so_far += 1
                ax = plt.subplot(3, 4, images_so_far)
                
                # Convertir imagen de tensor a numpy para visualización
                img = inputs[j].cpu().permute(1, 2, 0).numpy()
                img = (img * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
                img = np.clip(img, 0, 1)
                
                ax.imshow(img)
                
                # Colorear el título según la predicción sea correcta o no
                color = 'green' if preds[j] == labels[j] else 'red'
                ax.set_title(f'P: {class_names[preds[j]]}\nR: {class_names[labels[j]]}', color=color)
                plt.axis('off')
                
                if images_so_far == num_images:
                    plt.tight_layout()
                    return
        
        plt.tight_layout()

# Visualizar algunas predicciones
visualize_predictions(model, val_loader)

In [ ]:
# Guardar el modelo entrenado
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'class_names': class_names
}, 'beetle_classifier_resnet50.pth')

print("Modelo guardado como 'beetle_classifier_resnet50.pth'")

## Función para Predicción

A continuación se muestra cómo puedes utilizar el modelo entrenado para hacer predicciones en nuevas imágenes:

In [ ]:
# Función para hacer predicciones con una nueva imagen
def predict_image(image_path, model, class_names):
    # Cargar y preprocesar la imagen
    img = Image.open(image_path).convert('RGB')
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    img_tensor = transform(img).unsqueeze(0)
    
    # Mover al dispositivo correcto
    img_tensor = img_tensor.to(device)
    
    # Modo de evaluación
    model.eval()
    
    # Hacer predicción
    with torch.no_grad():
        outputs = model(img_tensor)
        _, preds = torch.max(outputs, 1)
        
        # Calcular probabilidades con softmax
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        top_probs, top_indices = torch.topk(probabilities, 3, dim=1)
        
    # Mostrar imagen y predicciones
    plt.figure(figsize=(8, 10))
    plt.subplot(2, 1, 1)
    plt.imshow(img)
    plt.title(f'Predicción: {class_names[preds.item()]}')
    plt.axis('off')
    
    # Visualizar las 3 clases con mayor probabilidad
    plt.subplot(2, 1, 2)
    top_probs = top_probs.squeeze().cpu().numpy()
    top_indices = top_indices.squeeze().cpu().numpy()
    
    top_classes = [class_names[i] for i in top_indices]
    
    plt.barh(range(3), top_probs, color='skyblue')
    plt.yticks(range(3), top_classes)
    plt.xlabel('Probabilidad')
    plt.title('Top 3 Predicciones')
    plt.tight_layout()
    plt.show()
    
    return preds.item(), probabilities.squeeze().cpu().numpy()

# Ejemplo de cómo usar la función de predicción (descomenta para probar)
# nueva_imagen = "ruta/a/nueva/imagen.jpg"
# pred_class, probabilities = predict_image(nueva_imagen, model, class_names)

## Conclusiones

En este notebook hemos:

1. Creado un modelo de clasificación de escarabajos utilizando una arquitectura ResNet50 preentrenada.
2. Aplicado técnicas de manejo de desbalance de clases para mejorar el rendimiento.
3. Implementado data augmentation para aumentar la diversidad de datos de entrenamiento.
4. Evaluado el modelo con métricas como precisión, recall, F1-score y matriz de confusión.
5. Visualizado las predicciones para entender mejor el rendimiento del modelo.

El modelo es capaz de clasificar escarabajos según su género, y puede ser utilizado para identificar automáticamente nuevos especímenes.

### Mejoras posibles

- Implementar ensembles de varios modelos para mejorar la precisión
- Probar con más arquitecturas (EfficientNet, Vision Transformers)
- Utilizar técnicas más avanzadas de data augmentation
- Recopilar más datos para clases con pocos ejemplos